## Running statistical tests to compare the sensitive attribute with other attributes:

### ccFraud

#### ccFraud Gender

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

col_names = ['gender', 'state', 'cardholder', 'balance', 'numTrans', 'numIntlTrans', 'creditLine', 'fraud']
df = pd.read_csv('/Users/himanshu/Documents/Projects/CALM-train-TrustworthyNLP/data/original_data/fraud detection/ccFraud/ccfraud_train.csv', names=col_names)

# Classify each attribute (excluding gender itself)
categorical_cols = ['state', 'cardholder', 'fraud']   # nominal / discrete-category vars
continuous_cols  = ['balance', 'numTrans', 'numIntlTrans', 'creditLine']  # numeric/skewed vars

results = []

# --- Categorical vs categorical: Chi-square test of independence ---
for col in categorical_cols:
    contingency = pd.crosstab(df['gender'], df[col])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    results.append({
        'attribute': col,
        'test': 'Chi-square',
        'statistic': chi2,
        'p_value': p,
        'significant (p<0.05)': p < 0.05
    })

# --- Categorical (gender) vs continuous: Point-biserial correlation + Mann-Whitney U ---
for col in continuous_cols:
    # Point-biserial correlation (Pearson r when one var is binary 0/1)
    gender_binary = df['gender'].map({1: 0, 2: 1})  # recode to 0/1 for the test
    r, p_corr = stats.pointbiserialr(gender_binary, df[col])

    # Mann-Whitney U as a robustness check (no normality assumption)
    group1 = df.loc[df['gender'] == 1, col]
    group2 = df.loc[df['gender'] == 2, col]
    u_stat, p_mw = stats.mannwhitneyu(group1, group2, alternative='two-sided')

    results.append({
        'attribute': col,
        'test': 'Point-biserial r',
        'statistic': r,
        'p_value': p_corr,
        'significant (p<0.05)': p_corr < 0.05
    })
    results.append({
        'attribute': col,
        'test': 'Mann-Whitney U',
        'statistic': u_stat,
        'p_value': p_mw,
        'significant (p<0.05)': p_mw < 0.05
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

   attribute             test     statistic  p_value  significant (p<0.05)
       state       Chi-square  5.334417e+01 0.346936                 False
  cardholder       Chi-square  1.944132e+00 0.163221                 False
       fraud       Chi-square  6.464982e+00 0.011002                  True
     balance Point-biserial r -2.502756e-02 0.032030                  True
     balance   Mann-Whitney U  6.535572e+06 0.017275                  True
    numTrans Point-biserial r  1.633307e-02 0.161790                 False
    numTrans   Mann-Whitney U  6.212478e+06 0.188803                 False
numIntlTrans Point-biserial r  6.019091e-03 0.606162                 False
numIntlTrans   Mann-Whitney U  6.387078e+06 0.456890                 False
  creditLine Point-biserial r -7.122574e-03 0.541809                 False
  creditLine   Mann-Whitney U  6.390810e+06 0.474276                 False


Why We Left `fraud (output label)` and `balance` Unadjusted During Counterfactual Generation (ccFraud, Gender)

**Statistical bias check.** Before generating gender counterfactuals for the ccFraud dataset, we ran correlation tests between gender and every other attribute in the dataset (chi-square for categorical fields, point-biserial correlation and Mann-Whitney U for continuous fields) to identify which fields might need adjustment alongside the sensitive attribute during augmentation.

**`fraud` (the label) — intentionally not adjusted, because it is the target of the intervention, not a confound.**

The test showed a statistically significant association between gender and the fraud label (χ² = 6.46, p = 0.011). This is expected and important, but it should not be treated the same way as a correlated input feature. `fraud` is the outcome variable the model predicts — a correlation between a sensitive attribute and the label is the definition of the bias we are trying to mitigate, not a data artifact to be corrected for realism. Our counterfactual augmentation strategy is built specifically to break this correlation: for each record, we generate a gender-flipped counterfactual while deliberately holding the label constant. This teaches the model that gender should not influence the fraud outcome. Adjusting the label to preserve the original correlation would be self-defeating — it would reproduce the exact bias the intervention is meant to remove.

**`balance` — statistically significant, but practically negligible, so left unadjusted with disclosure.**

`balance` showed a statistically significant relationship with gender (point-biserial r = -0.025, p = 0.032; Mann-Whitney U p = 0.017). However, an effect size of r ≈ -0.025 is negligible by standard interpretive thresholds (effects below ~0.1 are typically considered trivial). The significant p-values are best explained by the dataset's sample size (~10,000 records) rather than a meaningful underlying relationship — with large N, even tiny associations become statistically detectable despite carrying little practical weight. Adjusting `balance` during counterfactual generation to compensate for a correlation this small would add engineering complexity disproportionate to any realistic benefit, and risks introducing its own artifacts. We therefore leave `balance` unmodified when generating gender counterfactuals, while explicitly disclosing this decision and its statistical basis rather than silently ignoring it.

**All remaining fields** (`state`, `cardholder`, `numTrans`, `numIntlTrans`, `creditLine`) showed no statistically significant association with gender (all p > 0.05), supporting a clean, single-attribute swap for this dataset with no structural adjustments required.

**Takeaway for discussion**: the correlation check served two distinct purposes here — confirming that the label itself carries the bias we're targeting (which the augmentation directly addresses), and screening input features for confounds that would need separate handling (of which we found one candidate, `balance`, but determined its effect size did not warrant adjustment). Treating statistical significance and practical significance as separate questions — rather than reacting to any p < 0.05 result the same way — is the core judgment call documented here.

### Travel Insurance

#### Travel Insurance Age

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv('/Users/himanshu/Documents/Projects/CALM-train-TrustworthyNLP/data/original_data/insurance claim analysis/Travel Insurance/travel insurance.csv')

# --- Classify attributes relative to Age ---
continuous_cols   = ['Duration', 'Net Sales', 'Commision (in value)']
binary_cat_cols   = ['Agency Type', 'Distribution Channel', 'Claim', 'Gender']  # 2 levels (Gender has NaNs)
multilevel_cols   = ['Agency', 'Product Name', 'Destination']  # >2 levels, nominal

results = []

# --- Age vs continuous: Pearson + Spearman ---
for col in continuous_cols:
    sub = df[['Age', col]].dropna()
    r_p, p_p = stats.pearsonr(sub['Age'], sub[col])
    r_s, p_s = stats.spearmanr(sub['Age'], sub[col])
    results.append({'attribute': col, 'test': 'Pearson r', 'statistic': r_p, 'p_value': p_p})
    results.append({'attribute': col, 'test': 'Spearman rho', 'statistic': r_s, 'p_value': p_s})

# --- Age vs binary categorical: point-biserial + Mann-Whitney U ---
for col in binary_cat_cols:
    sub = df[['Age', col]].dropna()  # drops missing Gender rows automatically
    levels = sub[col].unique()
    if len(levels) != 2:
        print(f"Skipping {col}: found {len(levels)} levels, expected 2 -> {levels}")
        continue

    binary_map = {levels[0]: 0, levels[1]: 1}
    encoded = sub[col].map(binary_map)

    r_pb, p_pb = stats.pointbiserialr(encoded, sub['Age'])

    g1 = sub.loc[sub[col] == levels[0], 'Age']
    g2 = sub.loc[sub[col] == levels[1], 'Age']
    u_stat, p_mw = stats.mannwhitneyu(g1, g2, alternative='two-sided')

    results.append({'attribute': col, 'test': 'Point-biserial r', 'statistic': r_pb, 'p_value': p_pb})
    results.append({'attribute': col, 'test': 'Mann-Whitney U', 'statistic': u_stat, 'p_value': p_mw})

# --- Age vs multi-level nominal categorical: Kruskal-Wallis (+ ANOVA) ---
for col in multilevel_cols:
    sub = df[['Age', col]].dropna()
    groups = [g['Age'].values for _, g in sub.groupby(col)]
    groups = [g for g in groups if len(g) > 1]  # drop singleton groups

    h_stat, p_kw = stats.kruskal(*groups)
    f_stat, p_anova = stats.f_oneway(*groups)

    results.append({'attribute': col, 'test': 'Kruskal-Wallis H', 'statistic': h_stat, 'p_value': p_kw})
    results.append({'attribute': col, 'test': 'One-way ANOVA F', 'statistic': f_stat, 'p_value': p_anova})

results_df = pd.DataFrame(results)
results_df['significant (p<0.05)'] = results_df['p_value'] < 0.05
print(results_df.to_string(index=False))

           attribute             test     statistic       p_value  significant (p<0.05)
            Duration        Pearson r  2.468406e-03  5.344979e-01                 False
            Duration     Spearman rho -1.590991e-02  6.232000e-05                  True
           Net Sales        Pearson r  3.775601e-02  2.012585e-21                  True
           Net Sales     Spearman rho  2.349723e-02  3.345145e-09                  True
Commision (in value)        Pearson r  1.183483e-01 2.953428e-196                  True
Commision (in value)     Spearman rho  1.460044e-01 1.098795e-298                  True
         Agency Type Point-biserial r  2.215033e-01  0.000000e+00                  True
         Agency Type   Mann-Whitney U  3.368462e+08 6.712153e-222                  True
Distribution Channel Point-biserial r -1.596443e-01  0.000000e+00                  True
Distribution Channel   Mann-Whitney U  4.882472e+07 3.739154e-133                  True
               Claim Point-biser

In [3]:
import numpy as np
import pandas as pd
from scipy import stats

multilevel_cols = ['Agency', 'Product Name', 'Destination']
eta_results = []

for col in multilevel_cols:
    sub = df[['Age', col]].dropna()
    groups = [g['Age'].values for _, g in sub.groupby(col)]
    groups = [g for g in groups if len(g) > 1]  # drop singleton groups

    # --- Eta-squared from one-way ANOVA (SSB / SST) ---
    grand_mean = sub['Age'].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = sum((sub['Age'] - grand_mean) ** 2)
    eta_sq_anova = ss_between / ss_total

    # --- Eta-squared approximation from Kruskal-Wallis H ---
    # (rank-based, more robust when Age is skewed within groups)
    h_stat, p_kw = stats.kruskal(*groups)
    n = sum(len(g) for g in groups)
    k = len(groups)
    eta_sq_kw = (h_stat - k + 1) / (n - k)  # H-based epsilon-squared, commonly reported as eta^2_H

    eta_results.append({
        'attribute': col,
        'n_groups': k,
        'n_obs': n,
        'eta_sq (ANOVA)': eta_sq_anova,
        'eta_sq (Kruskal-Wallis approx)': eta_sq_kw
    })

eta_df = pd.DataFrame(eta_results)
print(eta_df.to_string(index=False))

   attribute  n_groups  n_obs  eta_sq (ANOVA)  eta_sq (Kruskal-Wallis approx)
      Agency        16  63326        0.450281                        0.127495
Product Name        25  63325        0.220792                        0.110479
 Destination       120  63297        0.115276                        0.019978


##### Summary of What the Statistical Results Actually Mean

- **Large dataset → p-values are misleading on their own.** With ~63,000 records, nearly every test came back "statistically significant" (p < 0.05), including relationships that are trivially small. So we ranked findings by **effect size** (r, η²) instead, which tells us how *strong* a relationship actually is, not just whether one exists.

- **Age vs. `Claim` (the label) is weak (r = -0.012) — and it doesn't matter either way.** This one isn't a confound to correct for realism; it's literally the bias signal the whole project is trying to address. We deliberately hold the label fixed when we flip age, regardless of how strong or weak this number is.

- **Most fields are negligible — safe to leave untouched.** Duration, Net Sales, and Gender all show weak-to-negligible correlation with age (r roughly 0.02–0.09). No adjustment needed when generating age-flipped counterfactuals.

- **Three fields show a real, moderate relationship with age: Agency, Agency Type, and Product Name.** These are business/product fields (which travel agency, which distribution channel, which insurance product), not personal traits of the traveler. That distinction matters: it's unclear that an older person flipped to younger should *also* get a different agency assigned — the association more likely reflects which products get marketed to or bought by different age groups, not age directly causing agency choice. Because of this, and because fixing three interacting categorical fields properly would be a much bigger modeling task than the project's scope allows, we chose to **leave them unadjusted and explicitly document the limitation** rather than build unjustified correction logic.

- **One test-selection catch worth noting: ANOVA and Kruskal-Wallis disagreed sharply for Agency and Destination** (e.g., Agency: 0.45 vs. 0.13). This is a sign ANOVA's assumptions (normal, equal-variance groups) don't hold well here — likely because Destination has 120 categories, many with very few records, and agency sales are probably skewed. We trusted the more robust Kruskal-Wallis numbers instead, which is why Destination ends up ranked as a small effect, not a medium one, despite what ANOVA alone would suggest.

**Bottom line**: age counterfactuals for this dataset are generated by flipping age across the 45-year threshold and holding all other fields constant — same simple approach as ccFraud — except now backed by evidence that most fields are safe to leave alone, with three specific business-channel fields flagged as a known, disclosed limitation rather than silently ignored.

##### Why We Left `Claim`, `Destination`, and Several Business-Channel Fields Unadjusted (or Flagged) During Counterfactual Generation (Travel Insurance, Age)

**Statistical bias check.** Before generating age counterfactuals for the Travel Insurance dataset, we tested the association between age and every other attribute — Pearson/Spearman correlation for continuous fields, point-biserial correlation and Mann-Whitney U for binary categorical fields, and both one-way ANOVA and Kruskal-Wallis (with corresponding eta-squared effect sizes) for multi-category fields. Given the dataset's large sample size (~63,000 records), we prioritized effect size over p-value throughout, since at this scale even trivial associations become statistically significant by conventional thresholds.

**`Claim` (the label) — intentionally not adjusted, because it is the target of the intervention, not a confound.**
Age showed a statistically detectable but practically negligible association with the claim label (point-biserial r = -0.012). As with any sensitive-attribute-to-label relationship, this is not treated as a confounding input feature to correct for realism — it is the outcome variable the debiasing intervention is designed to affect. Our counterfactual strategy generates age-flipped records (specifically crossing the 45-year threshold used in our fairness evaluation) while holding the claim label constant, directly targeting any learned dependence between age and outcome. Adjusting the label to preserve the original relationship would reproduce the bias rather than mitigate it.

**Negligible-effect fields — left unadjusted.**
Duration (Spearman rho = -0.016), Net Sales (Pearson r = 0.038), and Gender (point-biserial r = 0.086) all fall below or near the conventional 0.1 threshold for a "small" effect. We leave these fields unmodified when generating age counterfactuals; any statistical significance observed for them is attributable to sample size rather than a meaningful underlying relationship.

**Business-channel fields — genuine moderate association found, but deliberately not causally adjusted, and disclosed instead.**
Three fields showed real, non-trivial association with age: Agency Type (point-biserial r = 0.222), Agency (Kruskal-Wallis η² = 0.127), and Product Name (Kruskal-Wallis η² = 0.110). For the three multi-category fields (Agency, Product Name, Destination), we computed effect sizes under both ANOVA and Kruskal-Wallis and observed a substantial gap between the two for Agency and Destination in particular (e.g., Agency: η² = 0.450 under ANOVA vs. 0.127 under Kruskal-Wallis). This gap indicates that ANOVA's normality and equal-variance assumptions are likely violated here — plausible given Destination's 120 sparsely populated categories and the typically skewed distribution of insurance sales across agencies — so we treat the rank-based Kruskal-Wallis estimates as more reliable. Under that more trustworthy measure, Destination's association with age is small (η² = 0.020) and left unadjusted, while Agency and Product Name remain in the medium range and warrant explicit acknowledgment.

We chose not to build adjustment logic for Agency, Agency Type, or Product Name during counterfactual generation, for two reasons. First, these are business/product attributes rather than personal attributes of the insured individual — it is not clear that an age-flipped traveler should causally have a different agency or distribution channel, since these associations more plausibly reflect which products are marketed to or purchased by different age demographics than any direct effect of age itself. Second, correcting three interacting multi-category fields to preserve realistic joint distributions is a substantially larger undertaking (effectively a causal-modeling exercise) than is proportionate given project scope and timeline. We instead disclose this explicitly: age counterfactuals in this dataset do not adjust for Agency, Agency Type, or Product Name, which show medium effect sizes with age; this is a documented limitation rather than an unexamined gap.

**Takeaway for discussion.** This dataset required two additional judgment calls beyond the ccFraud analysis: (1) recognizing that with a large sample, statistical significance is a poor filter and effect size must drive decisions, and (2) checking whether a chosen statistical test's assumptions actually hold for the data (ANOVA vs. Kruskal-Wallis disagreement here), rather than trusting the first test run. Finding genuine, moderate confounds and choosing to disclose rather than force an unjustified causal correction is treated as the more defensible engineering decision given the project's scope.

### German Credit Scoring

In [7]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.contingency_tables import StratifiedTable

col_names = ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
             'savings', 'employment_since', 'installment_rate', 'personal_status_sex',
             'other_debtors', 'residence_since', 'property', 'age', 'other_installment_plans',
             'housing', 'existing_credits', 'job', 'num_liable', 'telephone', 'foreign_worker',
             'class']

df = pd.read_csv(
    '/Users/himanshu/Documents/Projects/CALM-train-TrustworthyNLP/data/original_data/credit_scoring/German/german.data',
    sep=r'\s+', names=col_names
)

# --- Derive gender from personal_status_sex (A91,A93,A94 = male; A92,A95 = female) ---
gender_map = {'A91': 'male', 'A93': 'male', 'A94': 'male', 'A92': 'female', 'A95': 'female'}
df['gender'] = df['personal_status_sex'].map(gender_map)

# --- Attribute type classification ---
continuous_cols = ['duration', 'credit_amount', 'installment_rate', 'residence_since',
                    'existing_credits', 'num_liable']
binary_cat_cols = ['other_debtors', 'other_installment_plans', 'housing', 'telephone',
                    'class']  # note: some of these have >2 levels below, reclassified
multilevel_cat_cols = ['checking_status', 'credit_history', 'purpose', 'savings',
                        'employment_since', 'property', 'job']

# fix: other_debtors(3), housing(3) are multi-level not binary; only telephone(2) and class(2) are truly binary
binary_cat_cols = ['telephone', 'class']
multilevel_cat_cols += ['other_debtors', 'other_installment_plans', 'housing']

focus_vars = {
    'gender': 'binary_cat',
    'age': 'continuous',
    'foreign_worker': 'binary_cat'
}

def run_test(focus_col, focus_type, other_col, other_type, data):
    sub = data[[focus_col, other_col]].dropna()
    result = {'focus': focus_col, 'attribute': other_col}

    if focus_type == 'binary_cat' and other_type == 'binary_cat':
        ct = pd.crosstab(sub[focus_col], sub[other_col])
        chi2, p, dof, exp = stats.chi2_contingency(ct)
        result.update(test='Chi-square', statistic=chi2, p_value=p)

    elif focus_type == 'binary_cat' and other_type == 'multilevel_cat':
        ct = pd.crosstab(sub[focus_col], sub[other_col])
        chi2, p, dof, exp = stats.chi2_contingency(ct)
        result.update(test='Chi-square', statistic=chi2, p_value=p)

    elif focus_type == 'binary_cat' and other_type == 'continuous':
        levels = sub[focus_col].unique()
        encoded = sub[focus_col].map({levels[0]: 0, levels[1]: 1})
        r, p = stats.pointbiserialr(encoded, sub[other_col])
        result.update(test='Point-biserial r', statistic=r, p_value=p)

    elif focus_type == 'continuous' and other_type == 'binary_cat':
        levels = sub[other_col].unique()
        encoded = sub[other_col].map({levels[0]: 0, levels[1]: 1})
        r, p = stats.pointbiserialr(encoded, sub[focus_col])
        result.update(test='Point-biserial r', statistic=r, p_value=p)

    elif focus_type == 'continuous' and other_type == 'multilevel_cat':
        groups = [g[focus_col].values for _, g in sub.groupby(other_col) if len(g) > 1]
        h, p = stats.kruskal(*groups)
        n, k = sum(len(g) for g in groups), len(groups)
        eta_sq = (h - k + 1) / (n - k)
        result.update(test='Kruskal-Wallis (eta_sq)', statistic=h, p_value=p, eta_sq=eta_sq)

    elif focus_type == 'continuous' and other_type == 'continuous':
        r, p = stats.spearmanr(sub[focus_col], sub[other_col])
        result.update(test='Spearman rho', statistic=r, p_value=p)

    return result

all_cols_typed = ({c: 'continuous' for c in continuous_cols} |
                   {c: 'binary_cat' for c in binary_cat_cols} |
                   {c: 'multilevel_cat' for c in multilevel_cat_cols})

results = []
for focus_col, focus_type in focus_vars.items():
    for other_col, other_type in all_cols_typed.items():
        if other_col == focus_col:
            continue
        # skip the other two focus vars if they overlap with candidate list (e.g. age itself)
        if other_col in ('age',) and focus_col in ('gender', 'foreign_worker'):
            other_type = 'continuous'
        results.append(run_test(focus_col, focus_type, other_col, other_type, df))

results_df = pd.DataFrame(results)
results_df['significant (p<0.05)'] = results_df['p_value'] < 0.05

#### German Gender

In [8]:
print(results_df[results_df['focus']=='gender'].to_string(index=False))

 focus               attribute             test  statistic      p_value  eta_sq  significant (p<0.05)
gender                duration Point-biserial r  -0.081432 9.990129e-03     NaN                  True
gender           credit_amount Point-biserial r  -0.093482 3.086806e-03     NaN                  True
gender        installment_rate Point-biserial r  -0.086302 6.318412e-03     NaN                  True
gender         residence_since Point-biserial r   0.013818 6.625208e-01     NaN                 False
gender        existing_credits Point-biserial r  -0.094260 2.848075e-03     NaN                  True
gender              num_liable Point-biserial r  -0.203431 8.412968e-11     NaN                  True
gender               telephone       Chi-square   5.440922 1.967028e-02     NaN                  True
gender                   class       Chi-square   5.348516 2.073991e-02     NaN                  True
gender         checking_status       Chi-square   0.741880 8.633119e-01     NaN   

#### German Age

In [9]:
print(results_df[results_df['focus']=='age'].to_string(index=False))

focus               attribute                    test  statistic      p_value    eta_sq  significant (p<0.05)
  age                duration            Spearman rho  -0.036316 2.512300e-01       NaN                 False
  age           credit_amount            Spearman rho   0.026298 4.061243e-01       NaN                 False
  age        installment_rate            Spearman rho   0.072157 2.249409e-02       NaN                  True
  age         residence_since            Spearman rho   0.234709 5.556995e-14       NaN                  True
  age        existing_credits            Spearman rho   0.141287 7.298173e-06       NaN                  True
  age              num_liable            Spearman rho   0.190651 1.222957e-09       NaN                  True
  age               telephone        Point-biserial r  -0.145259 3.983884e-06       NaN                  True
  age                   class        Point-biserial r  -0.091127 3.925339e-03       NaN                  True
  age     

#### German Foreign Status

In [10]:
print(results_df[results_df['focus']=='foreign_worker'].to_string(index=False))

         focus               attribute             test  statistic  p_value  eta_sq  significant (p<0.05)
foreign_worker                duration Point-biserial r  -0.138196 0.000012     NaN                  True
foreign_worker           credit_amount Point-biserial r  -0.050050 0.113711     NaN                 False
foreign_worker        installment_rate Point-biserial r  -0.090024 0.004385     NaN                  True
foreign_worker         residence_since Point-biserial r  -0.054097 0.087299     NaN                 False
foreign_worker        existing_credits Point-biserial r  -0.009717 0.758919     NaN                 False
foreign_worker              num_liable Point-biserial r   0.077071 0.014778     NaN                  True
foreign_worker               telephone       Chi-square  10.404570 0.001257     NaN                  True
foreign_worker                   class       Chi-square   5.821576 0.015831     NaN                  True
foreign_worker         checking_status       C